# hatchling: 5 Basic Usage Examples

In [ ]:
# Example 1: Basic hatchling import and version info
import hatchling

print(f"hatchling module available: {hatchling.__name__}")

# Check available builders
from hatchling.builders.wheel import WheelBuilder
from hatchling.builders.sdist import SdistBuilder
print(f"WheelBuilder: {WheelBuilder is not None}")
print(f"SdistBuilder: {SdistBuilder is not None}")

# Check metadata module
import hatchling.metadata.core
print(f"Metadata core module available: {hatchling.metadata.core is not None}")

assert (WheelBuilder.PLUGIN_NAME, SdistBuilder.PLUGIN_NAME) == ("wheel", "sdist")
assert hasattr(hatchling.metadata.core, "ProjectMetadata")
print("Hatchling basic imports completed")


In [ ]:
# Example 2: Create a project layout and build it into a wheel
import tempfile
import zipfile
from pathlib import Path

from hatchling.builders.wheel import WheelBuilder

pyproject_content = '''
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "test-hatch-pkg"
version = "0.1.0"
description = "Test package for hatchling"
'''

with tempfile.TemporaryDirectory() as temp:
    temp_dir = Path(temp)
    (temp_dir / "pyproject.toml").write_text(pyproject_content)

    # Create package directory
    pkg_dir = temp_dir / "test_hatch_pkg"
    pkg_dir.mkdir()
    (pkg_dir / "__init__.py").write_text('__version__ = "0.1.0"')
    print("Project structure created:", sorted(p.relative_to(temp_dir).as_posix() for p in temp_dir.rglob("*")))

    # build the wheel offline and look inside it
    (wheel_path,) = WheelBuilder(str(temp_dir)).build(directory=str(temp_dir / "dist"))
    assert Path(wheel_path).name == "test_hatch_pkg-0.1.0-py2.py3-none-any.whl", wheel_path
    with zipfile.ZipFile(wheel_path) as wheel:
        names = wheel.namelist()
        metadata = wheel.read("test_hatch_pkg-0.1.0.dist-info/METADATA").decode()
    print(f"Built {Path(wheel_path).name}: {names}")
    assert "test_hatch_pkg/__init__.py" in names
    assert "Name: test-hatch-pkg" in metadata and "Version: 0.1.0" in metadata


In [ ]:
# Example 3: Test hatchling metadata parsing
import tempfile
from pathlib import Path

import hatchling.metadata.core

pyproject_content = '''
[project]
name = "example-hatch"
version = "1.0.0"
description = "Example hatchling project"
'''

with tempfile.TemporaryDirectory() as temp:
    temp_dir = Path(temp)
    (temp_dir / "pyproject.toml").write_text(pyproject_content)

    metadata = hatchling.metadata.core.ProjectMetadata(str(temp_dir), None)
    assert (metadata.name, metadata.version, metadata.core.description) == ("example-hatch", "1.0.0", "Example hatchling project")
    print(f"Parsed metadata: {metadata.name} {metadata.version}")


In [ ]:
# Example 4: Build a wheel and an sdist
import tempfile
from pathlib import Path

from hatchling.builders.sdist import SdistBuilder
from hatchling.builders.wheel import WheelBuilder

pyproject_content = '''
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "build-test"
version = "0.1.0"
'''

with tempfile.TemporaryDirectory() as temp:
    temp_dir = Path(temp)
    (temp_dir / "pyproject.toml").write_text(pyproject_content)
    (temp_dir / "build_test.py").write_text('"""Build test module"""')

    # build the project above offline: a pure-Python wheel and an sdist, nothing fetched
    wheel_paths = list(WheelBuilder(str(temp_dir)).build(directory=str(temp_dir / "dist")))
    sdist_paths = list(SdistBuilder(str(temp_dir)).build(directory=str(temp_dir / "dist")))
    assert [Path(p).name for p in wheel_paths] == ["build_test-0.1.0-py2.py3-none-any.whl"], wheel_paths
    assert [Path(p).name for p in sdist_paths] == ["build_test-0.1.0.tar.gz"], sdist_paths
    print(f"Built: {Path(wheel_paths[0]).name}, {Path(sdist_paths[0]).name}")


In [ ]:
# Example 5: Explore hatchling plugins
try:
    import hatchling.plugin
    print(f"Plugin module available: {hatchling.plugin is not None}")
    
    # Check plugin submodules
    import pkgutil
    print("Plugin submodules:")
    for importer, modname, ispkg in pkgutil.iter_modules(hatchling.plugin.__path__, hatchling.plugin.__name__ + "."):
        print(f"  {modname}")
    
except Exception as e:
    print(f"Plugin exploration: {type(e).__name__}: {e}")
    raise AssertionError(
        f"hatchling example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

# Test build module
import hatchling.build
print(f"Build module available: {hatchling.build is not None}")
assert all(callable(getattr(hatchling.build, hook)) for hook in ("build_wheel", "build_sdist", "build_editable")), dir(hatchling.build)

print("Hatchling exploration completed")